In [1]:
# Resolve the CDR dataset and the workspace bucket
cdr    <- trimws(tail(system("wb resource resolve --id=C2024Q3R9", intern = TRUE), 1))
bucket <- trimws(tail(system("wb resource resolve --id=rw-migration-aou-rw-6f1dba72", intern = TRUE), 1))

cdr     # expect something like "project-id.C2024Q3R9" (exactly one dot)
bucket  # expect "gs://..."

Sys.setenv(WORKSPACE_CDR = cdr, WORKSPACE_BUCKET = bucket)

[1] "wb-silky-artichoke-2408.C2024Q3R9"

[1] "gs://rw-migration-aou-rw-6f1dba72"

In [30]:
BASE <- "/home/jupyter/workspace/rw-migration-aou-rw-6f1dba72/workspaces"
MISS <- file.path(BASE, "temporaldigitalhealthphenotyping/final_missing")
source(file.path(MISS, "load_missingness.R")); load_missingness(MISS)

EXPORT_DATE <- "20261011"            # the folder date of the 16 exports

L1_DIR  <- file.path(BASE, "result/final_missing/L1")
OUT_DIR <- file.path(BASE, "result/final_missing/results")
FINAL_PID   <- file.path(BASE, "result/final/final_pid.txt")
CCI_FILE    <- file.path(BASE, "result/pre_final/pre_final_cci.txt")
OVERLAP_POLICY <- "exclude"

cfg <- make_config(list(export  = list(date = EXPORT_DATE),
                        overlap = list(policy = OVERLAP_POLICY)))

cat("bucket :", cfg$export$bucket, "\nowner  :", cfg$export$owner, "\n")
cat("CDR    :", cfg$export$cdr, "\nbilling:", cfg$export$billing, "\n")
stopifnot("final_pid.txt not found - fix FINAL_PID"   = file.exists(FINAL_PID),
          "CCI file not found - fix CCI_FILE"         = file.exists(CCI_FILE),
          "WORKSPACE_BUCKET / OWNER_EMAIL not set"    = nzchar(cfg$export$bucket) && nzchar(cfg$export$owner))

bucket : gs://rw-migration-aou-rw-6f1dba72 
owner  : zqiu@researchallofus.org 
CDR    : wb-silky-artichoke-2408.C2024Q3R9 
billing: wb-dazzling-artichoke-8134 


In [31]:
need <- c("data.table", "lme4", "ordinal", "survival", "sandwich", "ggplot2", "rstatix", "ggpubr", "bigrquery")
print(sapply(need, requireNamespace, quietly = TRUE))   # any FALSE: install.packages("name")
#install.packages("sandwich")

data.table       lme4    ordinal   survival   sandwich    ggplot2    rstatix 
      TRUE       TRUE       TRUE       TRUE       TRUE       TRUE       TRUE 
    ggpubr  bigrquery 
      TRUE       TRUE 


In [4]:
cfg_demo <- make_config(list(boot = list(B = 50L)))
inp_demo <- simulate_inputs(cfg_demo)
res_demo <- run_missingness_pipeline(inp_demo$day_tables, inp_demo$persons, cfg_demo,
                                     file.path(tempdir(), "demo"),
                                     stages = c("missingness", "temporal", "descriptive", "onset"),
                                     onset_roles = "primary")
print(res_demo$primary_test[, .(stream, or = round(or, 2), p_holm)])    # expect or about 1.6-1.7 for all three

21:48:41 | hr: filled 6358 absent zone minute/calorie value(s) with 0

21:48:41 | hr: 1 day(s) have total zone minutes > 1440

Warning message:
“sleep: selection rule not fully met (n<21 days: 0; <70% observed: 4; gap>14 days: 0). Results are labelled as given.”
21:50:10 | [run] onset / activity / activity|joint|k1|adj  (session 0.6 GB, free 21.4 GB)

21:50:14 | [run] onset_extra / activity / vif  (session 0.6 GB, free 21.4 GB)

21:50:14 | [run] onset_extra / activity / cohort_interaction  (session 0.6 GB, free 21.4 GB)

21:50:21 | [run] onset_extra / activity / cohort_Insomnia  (session 0.6 GB, free 21.0 GB)

21:50:22 | [run] onset_extra / activity / cohort_Diabetes  (session 0.6 GB, free 21.2 GB)

21:50:24 | [run] onset_extra / activity / cohort_Healthy  (session 0.6 GB, free 21.1 GB)

21:50:26 | [run] onset_extra / activity / cohort_Prostate  (session 0.6 GB, free 21.1 GB)

21:50:26 | [run] onset_extra / activity / risk_unadjusted  (session 0.6 GB, free 21.4 GB)

21:50:29 | [run] on

     stream    or       p_holm
     <char> <num>        <num>
1: activity  1.59 6.207585e-22
2:       hr  1.73 2.115434e-41
3:    sleep  1.70 2.227767e-51


In [28]:
final_pid <- read_final_pid(FINAL_PID)
stopifnot("cohort names in final_pid must be Insomnia / Diabetes / Healthy / Prostate" =
            all(final_pid$cohort %in% cfg$cohorts))
print(final_pid[, .N, by = cohort])
ov <- resolve_cohort_overlap(final_pid, OVERLAP_POLICY, cfg$overlap$priority)
cat("unique people:", ov$n_unique_people, "| in more than one cohort:", length(ov$overlap_ids),
    "| kept with policy", OVERLAP_POLICY, ":", ov$n_kept, "\n")
print(ov$overlap)                      

     cohort     N
     <char> <int>
1: Diabetes  1298
2: Insomnia  1383
3: Prostate   250
4:  Healthy  6381
unique people: 8926 | in more than one cohort: 383 | kept with policy exclude : 8543 
                      combination n_people
                           <char>    <int>
1:            Diabetes + Insomnia      294
2:            Diabetes + Prostate       44
3:            Insomnia + Prostate       42
4: Diabetes + Insomnia + Prostate        3


In [34]:
# ==== CELL 6c   (OPTIONAL, separate from the exports above: the Fitbit device version of each person; nothing else is changed)
# 1) is there a device table with a device_version column?  2) does every person have a device version?  3) export it (4 files, one per cohort).
probe <- bq_probe_device_table(cfg)
print(probe$columns_check); print(probe$sample)
stopifnot("the device table has no device_version column (see probe$table_columns)" = probe$all_found)
dev_cov <- bq_device_coverage(cfg, final_pid)
print(bq_device_versions(cfg, final_pid), nrows = 100)   # which devices exist (device_type SCALE = a Fitbit scale, not a wearable)
dev_cov <- bq_device_coverage(cfg, final_pid)
print(dev_cov)

           column in_table
           <char>   <lgcl>
1:      person_id     TRUE
2:    device_date     TRUE
3: device_version     TRUE
4:    device_type     TRUE
   row_id person_id                            device_id device_date battery
1 3659538   7923631 3c946c98-1687-4443-86b9-814c1c672616  2023-04-02    <NA>
2 3057629   7603531 55955ae2-65aa-4696-89e8-308a668b285f        <NA>    <NA>
3 1113200   7943662 6e2c0078-a6a6-4c71-86f7-1d2426c1caba  2023-02-04    <NA>
4  697001   7459306 83afb2fa-6ed9-427d-9328-8f7f2ccca2cf        <NA>    <NA>
5 3510431   7318540 9172ae1c-7d80-4ef7-a46f-674c7e866361        <NA>    <NA>
  battery_level device_version device_type last_sync_time
1          <NA>       Aria Air       SCALE           <NA>
2          <NA>       Aria Air       SCALE           <NA>
3          <NA>       Aria Air       SCALE           <NA>
4          <NA>       Aria Air       SCALE           <NA>
5          <NA>       Aria Air       SCALE           <NA>
                    src_id
1

In [6]:
#Export from BigQuery test - test on PCA
run_bq_exports(cfg, final_pid, cohorts = "Prostate")

21:52:55 | export Prostate/activity: running query for 250 people

21:53:03 | export Prostate/sleep: running query for 250 people

21:53:09 | export Prostate/hr: running query for 250 people

21:53:15 | export Prostate/meta: running query for 250 people



     cohort   stream n_people   status      note
     <char>   <char>    <int>   <char>    <char>
1: Prostate activity      250 exported 1 file(s)
2: Prostate    sleep      250 exported 1 file(s)
3: Prostate       hr      250 exported 1 file(s)
4: Prostate     meta      250 exported 1 file(s)


In [13]:
exports <- run_bq_exports(cfg, final_pid)

21:57:41 | export Insomnia/activity: running query for 1383 people

21:57:51 | export Insomnia/sleep: running query for 1383 people

21:57:59 | export Insomnia/hr: running query for 1383 people

21:58:11 | export Insomnia/meta: running query for 1383 people

21:58:18 | export Diabetes/activity: running query for 1298 people

21:58:28 | export Diabetes/sleep: running query for 1298 people

21:58:36 | export Diabetes/hr: running query for 1298 people

21:58:49 | export Diabetes/meta: running query for 1298 people

21:58:54 | export Healthy/activity: running query for 6381 people

21:59:06 | export Healthy/sleep: running query for 6381 people

21:59:20 | export Healthy/hr: running query for 6381 people

21:59:56 | export Healthy/meta: running query for 6381 people

22:00:02 | export Prostate/activity: 1 file(s) already there - skipped

22:00:03 | export Prostate/sleep: 1 file(s) already there - skipped

22:00:05 | export Prostate/hr: 1 file(s) already there - skipped

22:00:06 | export Pr

      cohort   stream n_people   status               note
      <char>   <char>    <int>   <char>             <char>
 1: Insomnia activity     1383 exported          1 file(s)
 2: Insomnia    sleep     1383 exported          1 file(s)
 3: Insomnia       hr     1383 exported          1 file(s)
 4: Insomnia     meta     1383 exported          1 file(s)
 5: Diabetes activity     1298 exported          1 file(s)
 6: Diabetes    sleep     1298 exported          1 file(s)
 7: Diabetes       hr     1298 exported          1 file(s)
 8: Diabetes     meta     1298 exported          1 file(s)
 9:  Healthy activity     6381 exported          6 file(s)
10:  Healthy    sleep     6381 exported          1 file(s)
11:  Healthy       hr     6381 exported          1 file(s)
12:  Healthy     meta     6381 exported          1 file(s)
13: Prostate activity      250  skipped 1 existing file(s)
14: Prostate    sleep      250  skipped 1 existing file(s)
15: Prostate       hr      250  skipped 1 existing file(

In [14]:
for (co in cfg$cohorts) for (s in c("activity", "sleep", "hr", "meta")) {
  n <- tryCatch(length(gsutil_ls(export_path(cfg, co, s))), error = function(e) 0)
  cat(sprintf("%-9s %-9s %d file(s)\n", co, s, n))
}
# any "0 file(s)" = a missing export

Insomnia  activity  1 file(s)
Insomnia  sleep     1 file(s)
Insomnia  hr        1 file(s)
Insomnia  meta      1 file(s)
Diabetes  activity  1 file(s)
Diabetes  sleep     1 file(s)
Diabetes  hr        1 file(s)
Diabetes  meta      1 file(s)
Healthy   activity  6 file(s)
Healthy   sleep     1 file(s)
Healthy   hr        1 file(s)
Healthy   meta      1 file(s)
Prostate  activity  1 file(s)
Prostate  sleep     1 file(s)
Prostate  hr        1 file(s)
Prostate  meta      1 file(s)


In [5]:
#Build the compact tables (once per export
etl <- run_etl(cfg, L1_DIR,
               final_pid_path = file.path(BASE, "result/final/final_pid.txt"),
               cci_path       = file.path(BASE, "result/pre_final/pre_final_cci.txt"))
print(attr(etl$day_tables$activity, "dropped_columns"))
print(attr(etl$day_tables$sleep,    "dropped_columns"))
print(attr(etl$day_tables$hr,       "dropped_columns"))
print(etl$persons[, .N, by = cohort])  
print(etl$summary)                                   # rows / people / date range of each day table
print(etl$dropped_columns)                           # columns removed because they are mostly NA or constant
print(etl$persons[, .N, by = cohort])

06:35:50 | cohorts: 8926 people in final_pid, 383 belong to more than one cohort; policy 'exclude' keeps 8543



                      combination n_people
                           <char>    <int>
1:            Diabetes + Insomnia      294
2:            Diabetes + Prostate       44
3:            Insomnia + Prostate       42
4: Diabetes + Insomnia + Prostate        3


06:35:54 | reading 1 file(s) for insomnia_activity_*.csv

06:36:00 | reading 1 file(s) for diabetes_activity_*.csv

06:36:06 | reading 6 file(s) for healthy_activity_*.csv

06:36:29 | reading 1 file(s) for pca_activity_*.csv

06:37:00 | ETL activity: 4157233 rows, 8543 people, dropped columns: 

06:37:03 | reading 1 file(s) for insomnia_sleep.csv

06:37:06 | sleep: 5 day(s) with efficiency outside [0,100] set to NA

06:37:08 | reading 1 file(s) for diabetes_sleep.csv

06:37:12 | sleep: 1 day(s) with efficiency outside [0,100] set to NA

06:37:14 | reading 1 file(s) for healthy_sleep.csv

06:37:22 | sleep: 7 day(s) with efficiency outside [0,100] set to NA

06:37:24 | reading 1 file(s) for pca_sleep.csv

06:37:51 | ETL sleep: 3548818 rows, 8543 people, dropped columns: 

06:37:53 | reading 1 file(s) for insomnia_hr.csv

06:38:00 | hr: 1102 day(s) have total zone minutes > 1440

06:38:02 | reading 1 file(s) for diabetes_hr.csv

06:38:08 | hr: 848 day(s) have total zone minutes > 1440

06

NULL
NULL
NULL
     cohort     N
     <fctr> <int>
1:  Healthy  6381
2: Insomnia  1044
3: Diabetes   957
4: Prostate   161
     stream  n_rows n_persons n_columns
     <char>   <int>     <int>     <int>
1: activity 4157233      8543        14
2:    sleep 3548818      8543        16
3:       hr 3375496      8543        19
                                                                                                                                                                                                                                      columns
                                                                                                                                                                                                                                       <char>
1:                                        person_id,date,activity_calories,calories_bmr,calories_out,elevation,fairly_active_minutes,floors,lightly_active_minutes,marginal_calories,sedentary_minutes,st

In [6]:
#Check the heart-rate table, total_hr_min should be at most 1440
hr <- etl$day_tables$hr
hr_cols <- c("total_hr_min", "elevated_min", "mins_out_of_range", "mins_fat_burn",
                       "mins_cardio", "mins_peak", "total_cal")
print(summary(hr[, hr_cols]))

Length  Class   Mode 
     0   NULL   NULL 


In [32]:
# 10 Load the compact tables
inp <- load_inputs(cfg, L1_DIR)                      # nothing big is read: each stream is read when a stage needs it
print(data.frame(file = basename(inp$day_tables$paths), MB = round(file.size(inp$day_tables$paths) / 1e6, 1)))
cat("people:", nrow(inp$persons), "\n")
cat(sprintf("this R session: %.1f GB | free: %.1f GB | a model may use up to: %.1f GB\n",
            rss_gb(), available_memory_gb(), memory_headroom_gb(cfg)))

              file   MB
1 day_activity.rds 62.4
2    day_sleep.rds 58.3
3       day_hr.rds 98.2
4      persons.rds  0.1
people: 8543 
this R session: 2.4 GB | free: 13.4 GB | a model may use up to: 9.4 GB


In [45]:
# 11 a) quick part: cohort tables, temporal structure, descriptive statistics, figures
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR,
                                stages = c("missingness", "temporal", "descriptive"))
print(res$cohort_missingness)
print(res$selection_check)        # confirms the upstream filter (>=70% observed, gap <=14, >=21 days)

Warning message:
“Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes.”
Warning message:
“Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes.”
Warning message:
“Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes.”
Warning message:
“The `fatten` argument of `geom_boxplot()` is deprecated as of ggplot2 4.0.0.
ℹ Please use the `median.linewidth` argument instead.”
Warning message:
“Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes.”
Warning message:
“Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes.”
Warning message:
“Groups with fewer than two datapoints have been dropped.


      stream   cohort n_persons followup_days   record_days missing_pct
      <char>   <fctr>     <int>        <char>        <char>      <char>
 1: activity Insomnia      1044  113 (97-497)  112 (96-484)     0 (0-2)
 2: activity Diabetes       957  116 (99-474)  115 (97-472)     0 (0-2)
 3: activity  Healthy      6381 151 (102-741) 147 (101-736)     0 (0-2)
 4: activity Prostate       161  108 (95-442)  107 (93-425)     0 (0-1)
 5:       hr Insomnia      1044  111 (91-331)  102 (85-328)     3 (0-8)
 6:       hr Diabetes       957  112 (91-302)  104 (87-291)     3 (0-7)
 7:       hr  Healthy      6381  124 (91-635)  121 (88-617)     2 (0-6)
 8:       hr Prostate       161  107 (86-299)   96 (81-293)     3 (0-6)
 9:    sleep Insomnia      1044  108 (90-418)   95 (78-393)    7 (3-14)
10:    sleep Diabetes       957  111 (91-365)   96 (80-323)    7 (3-15)
11:    sleep  Healthy      6381  121 (91-645)  108 (82-605)    6 (2-14)
12:    sleep Prostate       161  102 (83-378)   93 (71-329)    5

In [46]:
# 11 missing % by sex / age group / race / CCI group with tests, and missing share by study week, year, weekday and age
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR, stages = "missing_figures")
print(res$missing_pct_by_stratum_tests[family == "groups_within_cohort" & test != "none",
      .(stream, stratum_var, cohort = group, test, n_total, statistic = round(statistic, 2), df, p = signif(p, 3),
        p_holm = signif(p_holm, 3), effect = round(effect, 3), effect_name)])

Warning message:
“Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes.”
Warning message:
“Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes.”
Warning message:
“Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes.”


      stream stratum_var   cohort              test n_total statistic    df
      <char>      <char>   <char>            <char>   <int>     <num> <num>
 1: activity         sex Insomnia Wilcoxon rank-sum    1040 113366.50    NA
 2: activity         sex Diabetes Wilcoxon rank-sum     956 111994.50    NA
 3: activity         sex  Healthy    Kruskal-Wallis    6381      0.43     2
 4:    sleep         sex Insomnia Wilcoxon rank-sum    1040 117630.00    NA
 5:    sleep         sex Diabetes Wilcoxon rank-sum     956 121804.50    NA
 6:    sleep         sex  Healthy    Kruskal-Wallis    6381      1.76     2
 7:       hr         sex Insomnia Wilcoxon rank-sum    1040 109156.00    NA
 8:       hr         sex Diabetes Wilcoxon rank-sum     956 109904.50    NA
 9:       hr         sex  Healthy    Kruskal-Wallis    6381     37.99     2
10: activity   age_group Insomnia    Kruskal-Wallis    1044     22.51     2
11: activity   age_group Diabetes    Kruskal-Wallis     957     10.80     2
12: activity

In [9]:
# 11c   (data quality: wear consistency, descriptive only - no model; ~1 min)
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR, stages = "wear_consistency",
                                model_streams = c("activity", "sleep", "hr"))
print(res$wear_consistency_classification[, .(stream, method, cutoff = round(threshold, 4), n_consistent, n_inconsistent)])
print(res$wear_consistency_descriptive[, .(stream, group, n_people, coverage_median = round(coverage_median, 3),
                                           gaps_per_100_observed_days = round(gaps_per_100_observed_days, 1), age_mean = round(age_mean, 1), pct_female = round(pct_female, 1))])
print(res$wear_consistency_composition[imbalanced == TRUE, .(stream, variable, smd = round(smd, 2))])   # characteristics that differ between the strata

     stream method cutoff n_consistent n_inconsistent
     <char> <char>  <num>        <int>          <int>
1: activity median 1.0000         5199           3344
2:    sleep median 0.9385         4272           4271
3:       hr median 0.9780         4382           4161
     stream        group n_people coverage_median gaps_per_100_observed_days
     <char>       <char>    <int>           <num>                      <num>
1: activity          All     8543           1.000                        0.7
2: activity   Consistent     5199           1.000                        0.0
3: activity Inconsistent     3344           0.978                        1.4
4:    sleep          All     8543           0.938                        3.7
5:    sleep   Consistent     4272           0.975                        1.8
6:    sleep Inconsistent     4271           0.862                        8.2
7:       hr          All     8543           0.978                        1.4
8:       hr   Consistent     4382    

In [10]:
# 12    b) the primary test and gap-definition models (the long part: probably around 3 hours)
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR, stages = "onset",
                                onset_roles = c("primary", "secondary_threshold"),
                                model_streams = "activity")
print(res$primary_test)

06:44:25 | [run] onset / activity / activity|joint|k1|adj  (session 1.4 GB, free 22.1 GB)

06:50:21 | [run] onset / activity / activity|joint|k2|adj  (session 2.7 GB, free 20.9 GB)

06:56:58 | [run] onset / activity / activity|joint|k3|adj  (session 2.7 GB, free 20.9 GB)

07:03:30 | [run] onset / activity / activity|joint|k7|adj  (session 2.7 GB, free 20.9 GB)

07:12:36 | [run] onset_extra / activity / vif  (session 2.7 GB, free 20.9 GB)

07:12:37 | [run] onset_extra / activity / cohort_interaction  (session 2.7 GB, free 20.9 GB)

07:25:01 | [run] onset_extra / activity / cohort_Insomnia  (session 2.7 GB, free 20.9 GB)

07:25:40 | [run] onset_extra / activity / cohort_Diabetes  (session 2.7 GB, free 20.9 GB)

07:26:13 | [run] onset_extra / activity / cohort_Healthy  (session 2.7 GB, free 20.9 GB)

07:29:50 | [run] onset_extra / activity / cohort_Prostate  (session 2.7 GB, free 20.9 GB)

07:29:57 | [run] onset_extra / activity / deciles  (session 2.7 GB, free 20.9 GB)

07:29:58 | [run] 

     stream        term   estimate        or    or_low   or_high     p p_holm
     <char>      <char>      <num>     <num>     <num>     <num> <num>  <num>
1: activity w_act_steps -0.4527874 0.6358533 0.6228594 0.6491183     0      0
   n_events status
      <int> <char>
1:    29712     ok


In [11]:
#13 gap on set model sleep
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR, stages = "onset",
                                onset_roles = c("primary", "secondary_threshold"),
                                model_streams = "sleep")
print(res$primary_test)

19:07:14 | [run] onset / sleep / sleep|joint|k1|adj  (session 1.9 GB, free 21.8 GB)



In [ ]:
# 14 gap onset model hr
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR, stages = "onset",
                                onset_roles = c("primary", "secondary_threshold"),
                                model_streams = "hr")
print(res$primary_test)

In [ ]:
# 15  Models B and C: 6 independent units per stream (4 x B, 2 x C), each saved on its own. They are built from
# the gaps and one activity predictor only (no person-day table), so they need little memory.
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR, stages = "gap_models",
                                model_streams = c("activity", "sleep", "hr"))
print(res$model_B_post_gap_activity[, .(stream, model_id, estimate = round(estimate, 3), p = signif(p, 3), status)])
print(res$model_C_gap_length[, .(stream, model_id, or = round(or, 2), po_p = signif(po_p, 2), status)])
cat("failed units:", nrow(res$unit_failures), "\n")      # 0 = nothing failed; otherwise run this cell again to retry them
if (nrow(res$unit_failures) > 0L) print(res$unit_failures)

In [ ]:
#16 sensitivity analysis
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR, stages = "sensitivity",
                                model_streams = c("activity", "sleep", "hr"))
print(res$sensitivity[, .(stream, analysis, term, or = round(or, 2), p = signif(p, 3), status)])

In [ ]:
# 17 exploratory on 3000 ppl
cfg_fast <- make_config(list(export  = list(date = EXPORT_DATE),
                             overlap = list(policy = OVERLAP_POLICY),
                             model   = list(exploratory_sample_persons = 3000L)))
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg_fast, OUT_DIR, stages = "onset",
                                onset_roles = "exploratory", model_streams = c("activity", "sleep", "hr"))
print(table(res$onset_models$role, res$onset_models$stream))

In [ ]:
# 18   (stratified analyses by wear consistency: the primary onset model, model B and model C within each stratum; long)
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR, stages = "wear_strata",
                                model_streams = c("activity", "sleep", "hr"))
print(res$wear_strata_comparison[, .(model, stream, consistent = round(effect_consistent, 2), inconsistent = round(effect_inconsistent, 2),
                                     p_difference = signif(p_difference, 3))])
print(res$wear_strata_quality[, .(model, stream, stratum, n_people, n_events, events_per_coefficient = round(events_per_coefficient, 1), status, sparse, usable)])
print(res$model_collinearity[figure == TRUE & flag != "ok", .(stream, analysis, term, vif_equiv = round(vif_equiv, 1), flag)])   # terms with a high / severe VIF-equivalent
cat("failed units:", nrow(res$unit_failures), "\n")

In [ ]:
#19 summarize and copy to busket
CKPT <- file.path(OUT_DIR, cfg$checkpoint$dirname)
print(checkpoint_report(CKPT, cfg)[, .N, by = .(kind, state)])     # valid / stale / failed / corrupt checkpoints
cat(readLines(file.path(OUT_DIR, "results_summary.md")), sep = "\n")
print(list.files(file.path(OUT_DIR, "tables")))
print(list.files(file.path(OUT_DIR, "figures")))                   # figure_captions.md explains every figure
system2("gsutil", c("-m", "cp", "-r", OUT_DIR, file.path(Sys.getenv("WORKSPACE_BUCKET"), "missingness_results/")))